# Training Data Cleaning And Diagnostics

Use this notebook to inspect `train.csv`, write a cleaned candidate CSV, and rerun diagnostics on the cleaned output.

The cleaning flow is intentionally narrow:
- canonicalize SVG serialization first
- resolve prompt conflicts by keeping one cleaned SVG per prompt
- write only a candidate CSV and report
- never modify the raw `train.csv`


## Environment Setup

This cell finds the project root, handles the Colab vs local path setup, and defines where the raw CSV, cleaned candidate CSV, and report file live.


In [6]:
import os
from pathlib import Path

try:
    from google.colab import drive
    IN_COLAB = True
except ModuleNotFoundError:
    drive = None
    IN_COLAB = False

project_root_override = os.environ.get('SVG_PROJECT_ROOT')

if project_root_override:
    PROJECT_ROOT = Path(project_root_override).expanduser().resolve()
elif IN_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/svg-kaggle-comptetition')
else:
    search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    PROJECT_ROOT = next(
        (root for root in search_roots if (root / 'train.csv').exists()),
        Path.cwd().resolve(),
    )

RAW_CSV_PATH = PROJECT_ROOT / 'train.csv'
DATASET_DIR = PROJECT_ROOT / 'datasets'
OUTPUT_DATASET_NAME = 'train_canonicalized_nosvgo_len2048.csv'
OUTPUT_REPORT_NAME = 'train_canonicalized_nosvgo_len2048_report.json'
OUTPUT_CANDIDATE_PATH = DATASET_DIR / OUTPUT_DATASET_NAME
REPORT_PATH = DATASET_DIR / OUTPUT_REPORT_NAME

DATASET_DIR.mkdir(parents=True, exist_ok=True)

print(f'Running in Colab: {IN_COLAB}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Raw CSV: {RAW_CSV_PATH}')
print(f'Candidate cleaned CSV: {OUTPUT_CANDIDATE_PATH}')
print(f'Cleaning report: {REPORT_PATH}')

if not RAW_CSV_PATH.exists():
    raise FileNotFoundError(
        f'Could not find train.csv at {RAW_CSV_PATH}. '
        'Set SVG_PROJECT_ROOT if your dataset lives somewhere else.'
    )


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Running in Colab: True
Project root: /content/drive/MyDrive/svg-kaggle-comptetition
Raw CSV: /content/drive/MyDrive/svg-kaggle-comptetition/train.csv
Candidate cleaned CSV: /content/drive/MyDrive/svg-kaggle-comptetition/datasets/train_canonicalized_nosvgo_len2048.csv
Cleaning report: /content/drive/MyDrive/svg-kaggle-comptetition/datasets/train_canonicalized_nosvgo_len2048_report.json


## Cleaning And Diagnostics Helpers

This cell defines the reusable helpers for loading rows, canonicalizing SVG text, computing diagnostics, and writing the cleaned CSV.


In [ ]:
import csv
import json
import random
import re
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

import pandas as pd
from IPython.display import display

TOKENIZER_MODEL_ID = 'Qwen/Qwen2.5-Coder-1.5B-Instruct'
SVG_NS = 'http://www.w3.org/2000/svg'
OUTPUT_COLUMNS = ['prompt', 'svg']
TRAINING_OBJECTIVE = 'prompt conditioning with SVG-only loss masking on the svg target'
MAX_SVG_LENGTH = 16000
MAX_PATH_COUNT = 256
RENDER_SIZE = 256
STRICT_VIEWBOX = f'0 0 {RENDER_SIZE} {RENDER_SIZE}'
ALLOWED_TAGS = {
    'svg', 'g', 'path', 'rect', 'circle', 'ellipse',
    'line', 'polyline', 'polygon', 'defs', 'use',
    'symbol', 'clipPath', 'mask', 'linearGradient',
    'radialGradient', 'stop', 'text', 'tspan',
    'title', 'desc', 'style', 'pattern', 'marker', 'filter',
}
COMMENT_RE = re.compile(r'<!--.*?-->', re.DOTALL)
FLOAT_RE = re.compile(r'(?<![A-Za-z])[-+]?(?:\d+\.\d+|\.\d+)(?:[eE][-+]?\d+)?')
EVENT_HANDLER_RE = re.compile(r'\son[a-zA-Z]+\s*=', re.IGNORECASE)
EXTERNAL_HREF_RE = re.compile(r"\s(?:href|xlink:href)\s*=\s*[\"']\s*(?:https?:|//)", re.IGNORECASE)
REMOTE_REF_RE = re.compile(r"@import\b|url\(\s*[\"']?(?:https?:|//)", re.IGNORECASE)
ROOT_TAG_RE = re.compile(r'<svg\b[^>]*>', flags=re.IGNORECASE | re.DOTALL)
def load_rows(csv_path: Path):
    with csv_path.open('r', encoding='utf-8', newline='') as handle:
        reader = csv.DictReader(handle)
        fieldnames = reader.fieldnames
        rows = list(reader)
    return fieldnames, rows


def format_training_text(prompt: str, svg: str) -> str:
    return f"Prompt: {prompt.strip()}\nSVG:\n{svg.strip()}"


def percentile(sorted_values, p):
    if not sorted_values:
        return None
    index = min(len(sorted_values) - 1, int((len(sorted_values) - 1) * p))
    return sorted_values[index]


def float_replacer(match, decimals):
    value = float(match.group(0))
    rounded = f"{value:.{decimals}f}".rstrip('0').rstrip('.')
    return '0' if rounded in {'', '-0'} else rounded


def strip_namespace(tag: str) -> str:
    return tag.split('}', 1)[-1] if '}' in tag else tag


def get_attr_value(opening_tag: str, attr_name: str):
    match = re.search(rf"\b{re.escape(attr_name)}\s*=\s*[\"']([^\"']+)[\"']", opening_tag, flags=re.IGNORECASE)
    if match is None:
        return None
    return match.group(1)


def upsert_attr(opening_tag: str, attr_name: str, attr_value: str) -> str:
    pattern = rf"(\b{re.escape(attr_name)}\s*=\s*)([\"'])(.*?)(\2)"
    if re.search(pattern, opening_tag, flags=re.IGNORECASE | re.DOTALL):
        def replacer(match):
            prefix, quote, _, suffix = match.groups()
            return f'{prefix}{quote}{attr_value}{suffix}'

        return re.sub(pattern, replacer, opening_tag, count=1, flags=re.IGNORECASE | re.DOTALL)
    return opening_tag[:-1] + f' {attr_name}="{attr_value}">'


def normalize_root_attributes(svg: str) -> str:
    match = ROOT_TAG_RE.search(svg)
    if match is None:
        return svg

    opening_tag = match.group(0)
    normalized_tag = (
        f'<svg xmlns="{SVG_NS}" width="{RENDER_SIZE}" height="{RENDER_SIZE}" '
        f'viewBox="{STRICT_VIEWBOX}">'
    )
    return svg.replace(opening_tag, normalized_tag, 1)


def canonicalize_svg(
    svg: str,
    strip_comments=True,
    remove_filling_attr=True,
    round_floats=True,
    float_decimals=3,
    collapse_whitespace=True,
):
    cleaned = svg.strip()
    if strip_comments:
        cleaned = COMMENT_RE.sub('', cleaned)
    if remove_filling_attr:
        cleaned = re.sub(r'\s*filling="[^"]*"', '', cleaned)
        cleaned = re.sub(r"\s*filling='[^']*'", '', cleaned)
    if round_floats:
        cleaned = FLOAT_RE.sub(lambda match: float_replacer(match, float_decimals), cleaned)
    if collapse_whitespace:
        cleaned = re.sub(r'>\s+<', '><', cleaned)
        cleaned = re.sub(r'\s+', ' ', cleaned)
    return cleaned.strip()


def midterm_gate_reason(svg: str):
    if not isinstance(svg, str) or not svg.strip():
        return 'svg_too_long_or_empty'

    svg = svg.strip()
    if len(svg) > MAX_SVG_LENGTH:
        return 'svg_too_long_or_empty'
    if EVENT_HANDLER_RE.search(svg):
        return 'disallowed_attr:event_handler'
    if EXTERNAL_HREF_RE.search(svg):
        return 'disallowed_ref:external_href'
    if REMOTE_REF_RE.search(svg):
        return 'disallowed_ref:remote_url'

    opening_match = ROOT_TAG_RE.search(svg)
    if opening_match is None:
        return 'strict_parse_failed'
    opening_tag = opening_match.group(0)
    if get_attr_value(opening_tag, 'xmlns') != SVG_NS:
        return 'missing_xmlns'

    try:
        root = ET.fromstring(svg)
    except Exception:
        return 'parse_failed'

    if strip_namespace(root.tag) != 'svg':
        return 'root_not_svg'
    if root.attrib.get('viewBox') != STRICT_VIEWBOX:
        return 'viewbox_not_exact'
    if root.attrib.get('width') != str(RENDER_SIZE) or root.attrib.get('height') != str(RENDER_SIZE):
        return 'width_height_not_exact'

    path_count = 0
    for elem in root.iter():
        tag = strip_namespace(elem.tag)
        if tag not in ALLOWED_TAGS:
            return f'disallowed_tag:{tag}'
        if tag == 'path':
            path_count += 1

    if path_count > MAX_PATH_COUNT:
        return 'too_many_paths'
    return 'valid'


def run_diagnostics(csv_path: Path):
    fieldnames, rows = load_rows(csv_path)

    prompt_counter = Counter()
    pair_counter = Counter()
    prompt_to_svgs = defaultdict(set)
    svg_to_prompts = defaultdict(set)

    parse_fail = 0
    non_svg_root = 0
    style_tag = 0
    comments = 0
    current_color = 0
    filling_attr = 0
    high_precision_numbers = 0
    missing_viewbox = 0
    midterm_xmlns_exact = 0
    midterm_viewbox_exact = 0
    midterm_width_height_exact = 0
    midterm_root_exact = 0
    midterm_gate_fail = 0

    prompt_lengths = []
    svg_lengths = []
    formatted_lengths = []

    for row in rows:
        prompt = '' if row.get('prompt') is None else str(row['prompt']).strip()
        svg = '' if row.get('svg') is None else str(row['svg']).strip()

        prompt_counter[prompt] += 1
        pair_counter[(prompt, svg)] += 1
        prompt_to_svgs[prompt].add(svg)
        svg_to_prompts[svg].add(prompt)

        prompt_lengths.append(len(prompt))
        svg_lengths.append(len(svg))
        formatted_lengths.append(len(format_training_text(prompt, svg)))

        lower = svg.lower()
        if '<style' in lower:
            style_tag += 1
        if '<!--' in svg:
            comments += 1
        if 'currentColor' in svg:
            current_color += 1
        if 'filling=' in svg:
            filling_attr += 1
        if re.search(r'\d+\.\d{4,}', svg):
            high_precision_numbers += 1

        opening_match = ROOT_TAG_RE.search(svg)
        if opening_match is not None:
            opening_tag = opening_match.group(0)
            if get_attr_value(opening_tag, 'xmlns') == SVG_NS:
                midterm_xmlns_exact += 1

        try:
            root = ET.fromstring(svg)
            if strip_namespace(root.tag) != 'svg':
                non_svg_root += 1
            if 'viewBox' not in root.attrib:
                missing_viewbox += 1
            if root.attrib.get('viewBox') == STRICT_VIEWBOX:
                midterm_viewbox_exact += 1
            if root.attrib.get('width') == str(RENDER_SIZE) and root.attrib.get('height') == str(RENDER_SIZE):
                midterm_width_height_exact += 1
            if opening_match is not None and get_attr_value(opening_tag, 'xmlns') == SVG_NS and root.attrib.get('viewBox') == STRICT_VIEWBOX and root.attrib.get('width') == str(RENDER_SIZE) and root.attrib.get('height') == str(RENDER_SIZE):
                midterm_root_exact += 1
        except Exception:
            parse_fail += 1

        if midterm_gate_reason(svg) != 'valid':
            midterm_gate_fail += 1

    exact_duplicate_rows = sum(count - 1 for count in pair_counter.values() if count > 1)
    conflicting_prompts = {prompt for prompt, svgs in prompt_to_svgs.items() if len(svgs) > 1}
    conflicting_rows = sum(prompt_counter[prompt] for prompt in conflicting_prompts)
    reused_svg_count = sum(1 for prompts in svg_to_prompts.values() if len(prompts) > 1)

    prompt_lengths = sorted(prompt_lengths)
    svg_lengths = sorted(svg_lengths)
    formatted_lengths = sorted(formatted_lengths)

    summary = {
        'csv_path': str(csv_path),
        'rows': len(rows),
        'columns': fieldnames,
        'exact_duplicate_rows': exact_duplicate_rows,
        'conflicting_prompt_count': len(conflicting_prompts),
        'conflicting_rows': conflicting_rows,
        'conflicting_row_pct': round(conflicting_rows / len(rows) * 100, 2) if rows else 0.0,
        'reused_svg_count': reused_svg_count,
        'parse_fail': parse_fail,
        'non_svg_root': non_svg_root,
        'missing_viewBox': missing_viewbox,
        'style_tag_rows': style_tag,
        'comment_rows': comments,
        'currentColor_rows': current_color,
        'filling_attr_rows': filling_attr,
        'high_precision_rows': high_precision_numbers,
        'midterm_xmlns_exact_rows': midterm_xmlns_exact,
        'midterm_viewBox_exact_rows': midterm_viewbox_exact,
        'midterm_width_height_exact_rows': midterm_width_height_exact,
        'midterm_root_exact_rows': midterm_root_exact,
        'midterm_gate_fail_rows': midterm_gate_fail,
        'prompt_chars_p50': percentile(prompt_lengths, 0.5),
        'prompt_chars_p95': percentile(prompt_lengths, 0.95),
        'prompt_chars_p99': percentile(prompt_lengths, 0.99),
        'svg_chars_p50': percentile(svg_lengths, 0.5),
        'svg_chars_p95': percentile(svg_lengths, 0.95),
        'svg_chars_p99': percentile(svg_lengths, 0.99),
        'svg_chars_max': svg_lengths[-1] if svg_lengths else None,
        'formatted_chars_p50': percentile(formatted_lengths, 0.5),
        'formatted_chars_p95': percentile(formatted_lengths, 0.95),
        'formatted_chars_p99': percentile(formatted_lengths, 0.99),
    }

    top_conflicts = pd.DataFrame(
        sorted(
            [
                {'count': prompt_counter[prompt], 'unique_svgs': len(svgs), 'prompt': prompt[:220]}
                for prompt, svgs in prompt_to_svgs.items() if len(svgs) > 1
            ],
            key=lambda row: (-row['unique_svgs'], -row['count'], row['prompt']),
        )[:20]
    )

    longest_rows = pd.DataFrame(
        sorted(
            [
                {'prompt': str(row['prompt'])[:180], 'svg_chars': len(str(row['svg'])), 'svg_prefix': str(row['svg'])[:220].replace('\n', ' ')}
                for row in rows
            ],
            key=lambda row: -row['svg_chars'],
        )[:20]
    )

    print(f'CSV: {csv_path}')
    display(pd.Series(summary, name='value').to_frame())
    if not top_conflicts.empty:
        display(top_conflicts)
    display(longest_rows)

    return {
        'fieldnames': fieldnames,
        'rows': rows,
        'summary': summary,
        'top_conflicts': top_conflicts,
        'longest_rows': longest_rows,
    }


def tokenize_texts(texts, model_id=TOKENIZER_MODEL_ID, batch_size=128):
    from transformers import AutoTokenizer

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    token_lengths = []
    for start in range(0, len(texts), batch_size):
        batch = texts[start:start + batch_size]
        encoded = tokenizer(batch, add_special_tokens=True, padding=False, truncation=False)
        token_lengths.extend(len(ids) for ids in encoded['input_ids'])
    return token_lengths


def tokenize_rows(rows, model_id=TOKENIZER_MODEL_ID, batch_size=128):
    texts = [format_training_text(str(row['prompt']), str(row['svg'])) for row in rows]
    return tokenize_texts(texts, model_id=model_id, batch_size=batch_size)


def run_token_length_diagnostics(csv_path: Path, model_id=TOKENIZER_MODEL_ID, sample_n=2000, seed=42, max_length=2048):
    _, rows = load_rows(csv_path)
    if not rows:
        return None

    sample_rows = rows if sample_n is None or sample_n >= len(rows) else random.Random(seed).sample(rows, sample_n)
    token_lengths = sorted(tokenize_rows(sample_rows, model_id=model_id))
    summary = {
        'token_model_id': model_id,
        'token_sample_n': len(token_lengths),
        'token_p50': percentile(token_lengths, 0.5),
        'token_p95': percentile(token_lengths, 0.95),
        'token_p99': percentile(token_lengths, 0.99),
        'token_max': token_lengths[-1],
        'over_max_length': sum(length > max_length for length in token_lengths),
        'over_max_length_pct': round(sum(length > max_length for length in token_lengths) / len(token_lengths) * 100, 2),
        'max_length': max_length,
    }
    display(pd.Series(summary, name='value').to_frame())
    return summary


def filter_rows_by_token_budget(rows, max_length=2048, model_id=TOKENIZER_MODEL_ID):
    if not rows:
        return rows, {
            'token_filter_model_id': model_id,
            'token_filter_max_length': max_length,
            'rows_evaluated_for_token_budget': 0,
            'dropped_rows_over_token_budget': 0,
            'kept_rows_after_token_budget': 0,
        }

    token_lengths = tokenize_rows(rows, model_id=model_id)
    kept_rows = []
    dropped = 0
    over_lengths = []
    for row, token_length in zip(rows, token_lengths):
        if token_length > max_length:
            dropped += 1
            over_lengths.append(token_length)
            continue
        kept_rows.append(row)

    report = {
        'token_filter_model_id': model_id,
        'token_filter_max_length': max_length,
        'rows_evaluated_for_token_budget': len(rows),
        'dropped_rows_over_token_budget': dropped,
        'kept_rows_after_token_budget': len(kept_rows),
        'token_filter_over_budget_pct': round(dropped / len(rows) * 100, 2),
    }
    if over_lengths:
        over_lengths = sorted(over_lengths)
        report['token_filter_over_budget_p50'] = percentile(over_lengths, 0.5)
        report['token_filter_over_budget_p95'] = percentile(over_lengths, 0.95)
        report['token_filter_over_budget_max'] = over_lengths[-1]
    return kept_rows, report


def build_cleaned_rows(
    raw_rows,
    strip_comments=True,
    remove_filling_attr=True,
    round_floats=True,
    float_decimals=3,
    collapse_whitespace=True,
    normalize_midterm_root=True,
    enforce_midterm_gate=True,
    conflict_policy='keep_most_frequent',
    max_svg_chars=MAX_SVG_LENGTH,
    deduplicate_exact_pairs=True,
):
    if conflict_policy not in {'keep_all', 'keep_most_frequent', 'drop'}:
        raise ValueError(f'Unsupported conflict_policy: {conflict_policy}')

    normalized_rows = []
    report = Counter()
    report['input_rows'] = len(raw_rows)

    for row in raw_rows:
        prompt = '' if row.get('prompt') is None else str(row['prompt']).strip()
        svg = '' if row.get('svg') is None else str(row['svg']).strip()

        if not prompt or not svg:
            report['dropped_empty_rows'] += 1
            continue

        cleaned_svg = canonicalize_svg(
            svg,
            strip_comments=strip_comments,
            remove_filling_attr=remove_filling_attr,
            round_floats=round_floats,
            float_decimals=float_decimals,
            collapse_whitespace=collapse_whitespace,
        )

        try:
            root = ET.fromstring(cleaned_svg)
            if strip_namespace(root.tag) != 'svg':
                report['dropped_non_svg_root_rows'] += 1
                continue
        except Exception:
            report['dropped_parse_fail_rows'] += 1
            continue

        if cleaned_svg != svg:
            report['rows_changed_by_svg_cleaning'] += 1

        normalized_rows.append({'prompt': prompt, 'svg': cleaned_svg})

    for row in normalized_rows:
        if normalize_midterm_root:
            normalized_root_svg = normalize_root_attributes(row['svg'])
            if normalized_root_svg != row['svg']:
                report['rows_changed_by_midterm_root_normalization'] += 1
            row['svg'] = normalized_root_svg

    filtered_rows = []
    for row in normalized_rows:
        svg = row['svg']
        if max_svg_chars is not None and len(svg) > max_svg_chars:
            report['dropped_long_svg_rows'] += 1
            continue
        if enforce_midterm_gate:
            gate_reason = midterm_gate_reason(svg)
            if gate_reason != 'valid':
                report['dropped_midterm_gate_rows'] += 1
                report[f'midterm_gate__{gate_reason}'] += 1
                continue
        filtered_rows.append(row)

    prompt_svg_counts = defaultdict(Counter)
    for row in filtered_rows:
        prompt_svg_counts[row['prompt']][row['svg']] += 1

    conflicting_prompts = {
        prompt: svg_counts
        for prompt, svg_counts in prompt_svg_counts.items()
        if len(svg_counts) > 1
    }
    chosen_svg_by_prompt = {}
    if conflict_policy == 'keep_most_frequent':
        for prompt, svg_counts in conflicting_prompts.items():
            chosen_svg_by_prompt[prompt] = min(svg_counts.items(), key=lambda item: (-item[1], len(item[0]), item[0]))[0]

    kept_rows = []
    seen_pairs = set()
    for row in filtered_rows:
        prompt = row['prompt']
        svg = row['svg']

        if conflict_policy == 'drop' and prompt in conflicting_prompts:
            report['dropped_conflicting_prompt_rows'] += 1
            continue
        if conflict_policy == 'keep_most_frequent' and prompt in chosen_svg_by_prompt and svg != chosen_svg_by_prompt[prompt]:
            report['dropped_rows_from_conflict_resolution'] += 1
            continue

        cleaned_pair = (prompt, svg)
        if deduplicate_exact_pairs and cleaned_pair in seen_pairs:
            report['dropped_exact_duplicate_pairs_after_cleaning'] += 1
            continue
        seen_pairs.add(cleaned_pair)
        kept_rows.append({'prompt': prompt, 'svg': svg})

    report['kept_rows'] = len(kept_rows)
    report['normalize_midterm_root'] = bool(normalize_midterm_root)
    report['enforce_midterm_gate'] = bool(enforce_midterm_gate)
    report['conflict_policy'] = conflict_policy
    report['conflicting_prompts_after_cleaning'] = len(conflicting_prompts)
    report['resolved_conflicting_prompt_count'] = len(chosen_svg_by_prompt)
    report['max_svg_chars'] = max_svg_chars
    return kept_rows, dict(report)


def write_cleaned_csv(csv_path: Path, cleaned_rows):
    with csv_path.open('w', encoding='utf-8', newline='') as handle:
        writer = csv.DictWriter(handle, fieldnames=OUTPUT_COLUMNS)
        writer.writeheader()
        writer.writerows(cleaned_rows)


## Raw Dataset Diagnostics

This cell runs the diagnostics on the raw `train.csv` so you can inspect the starting point before any cleaning or conflict resolution is applied.


In [8]:
raw_results = run_diagnostics(RAW_CSV_PATH)

CSV: /content/drive/MyDrive/svg-kaggle-comptetition/train.csv


,value
csv_path,/content/drive/MyDrive/svg-kaggle-comptetition...
rows,50000
columns,"[id, prompt, svg]"
exact_duplicate_rows,0
conflicting_prompt_count,1071
conflicting_rows,5140
conflicting_row_pct,10.28
reused_svg_count,7
parse_fail,0
non_svg_root,0


,count,unique_svgs,prompt
0,1788,1788,Generate svg code for an image that looks like...
1,34,34,Generate svg code for an image that looks like...
2,33,33,The image is completely filled with a solid bl...
3,24,24,Generate svg code for an image that looks like...
4,21,21,A simple black cross symbol on a white backgro...
5,20,20,The image consists of a single solid black col...
6,19,19,"The image consists of a single solid color, wh..."
7,17,17,A black arrow pointing to the right on a white...
8,17,17,A simple gray arrow pointing to the right.
9,17,17,A simple gray magnifying glass icon.


,prompt,svg_chars,svg_prefix
0,A pair of pruning shears is shown in a black o...,15937,"<svg height=""128"" viewBox=""0 -27.61 133.31 133..."
1,A cartoon character with brown hair is shruggi...,15881,"<svg fill=""none"" height=""128"" viewBox=""0 0 128..."
2,"A line drawing of a bird, likely a bird of pre...",15833,"<svg height=""128"" viewBox=""-12.59 0 119.14 119..."
3,A golden ring with a large blue gemstone accen...,15583,"<svg fill=""none"" height=""128"" viewBox=""0 0 128..."
4,Figure 1 - Key and Seal,15499,"<svg height=""128"" viewBox=""0 0 32 32"" width=""1..."
5,A stylized cartoon explosion with red and yell...,15468,"<svg fill=""none"" height=""128"" viewBox=""0 0 128..."
6,The image shows three hands next to each other...,15260,"<svg enable-background=""new 0 0 512 512"" heigh..."
7,A purple diamond-shaped icon with three square...,15163,"<svg height=""128"" viewBox=""0 0 1024 1024"" widt..."
8,A black and white icon of a folder with a worl...,15038,"<svg height=""128"" preserveAspectRatio=""xMidYMi..."
9,"A bedroom with a bed, desk with a computer, an...",14541,"<svg height=""128"" viewBox=""0 -66.86 410.78 410..."


## Cleaning Configuration And Candidate Build

This cell sets the cleaning knobs, keeps the cleaned dataset in separate `prompt` and `svg` columns for SVG-only loss masking during training, filters rows that exceed an exact token budget, resolves prompt conflicts, and writes the candidate CSV plus cleaning report.


In [9]:
STRIP_COMMENTS = True
REMOVE_FILLING_ATTR = True
ROUND_FLOATS = True
FLOAT_DECIMALS = 3
COLLAPSE_WHITESPACE = True
TRAIN_MAX_LENGTH = 2048
NORMALIZE_MIDTERM_ROOT = True
ENFORCE_MIDTERM_GATE = True
CONFLICT_POLICY = 'keep_most_frequent'  # Options: keep_all, keep_most_frequent, drop
MAX_SVG_CHARS = MAX_SVG_LENGTH
DEDUPLICATE_EXACT_PAIRS = True
FILTER_OVER_TOKEN_BUDGET = True
TOKEN_FILTER_MODEL_ID = TOKENIZER_MODEL_ID
TOKEN_FILTER_MAX_LENGTH = TRAIN_MAX_LENGTH
TOKEN_SAMPLE_N = 2000
LOSS_MASKING_READY = True

print({
    'STRIP_COMMENTS': STRIP_COMMENTS,
    'REMOVE_FILLING_ATTR': REMOVE_FILLING_ATTR,
    'ROUND_FLOATS': ROUND_FLOATS,
    'FLOAT_DECIMALS': FLOAT_DECIMALS,
    'COLLAPSE_WHITESPACE': COLLAPSE_WHITESPACE,
    'TRAIN_MAX_LENGTH': TRAIN_MAX_LENGTH,
    'NORMALIZE_MIDTERM_ROOT': NORMALIZE_MIDTERM_ROOT,
    'ENFORCE_MIDTERM_GATE': ENFORCE_MIDTERM_GATE,
    'STRICT_VIEWBOX': STRICT_VIEWBOX,
    'RENDER_SIZE': RENDER_SIZE,
    'MAX_PATH_COUNT': MAX_PATH_COUNT,
    'MAX_SVG_CHARS': MAX_SVG_CHARS,
    'CONFLICT_POLICY': CONFLICT_POLICY,
    'DEDUPLICATE_EXACT_PAIRS': DEDUPLICATE_EXACT_PAIRS,
    'FILTER_OVER_TOKEN_BUDGET': FILTER_OVER_TOKEN_BUDGET,
    'TOKEN_FILTER_MODEL_ID': TOKEN_FILTER_MODEL_ID,
    'TOKEN_FILTER_MAX_LENGTH': TOKEN_FILTER_MAX_LENGTH,
    'TOKEN_SAMPLE_N': TOKEN_SAMPLE_N,
    'OUTPUT_COLUMNS': OUTPUT_COLUMNS,
    'LOSS_MASKING_READY': LOSS_MASKING_READY,
    'TRAINING_OBJECTIVE': TRAINING_OBJECTIVE,
})

_, raw_rows = load_rows(RAW_CSV_PATH)

cleaned_rows, cleaning_report = build_cleaned_rows(
    raw_rows,
    strip_comments=STRIP_COMMENTS,
    remove_filling_attr=REMOVE_FILLING_ATTR,
    round_floats=ROUND_FLOATS,
    float_decimals=FLOAT_DECIMALS,
    collapse_whitespace=COLLAPSE_WHITESPACE,
    normalize_midterm_root=NORMALIZE_MIDTERM_ROOT,
    enforce_midterm_gate=ENFORCE_MIDTERM_GATE,
    conflict_policy=CONFLICT_POLICY,
    max_svg_chars=MAX_SVG_CHARS,
    deduplicate_exact_pairs=DEDUPLICATE_EXACT_PAIRS,
)

if FILTER_OVER_TOKEN_BUDGET:
    cleaned_rows, token_filter_report = filter_rows_by_token_budget(
        cleaned_rows,
        max_length=TOKEN_FILTER_MAX_LENGTH,
        model_id=TOKEN_FILTER_MODEL_ID,
    )
    cleaning_report.update(token_filter_report)

cleaning_report['output_columns'] = OUTPUT_COLUMNS
cleaning_report['loss_masking_ready'] = LOSS_MASKING_READY
cleaning_report['training_objective'] = TRAINING_OBJECTIVE
cleaning_report['training_input_contract'] = 'use prompt as context and apply loss only on svg tokens during fine-tuning'

write_cleaned_csv(OUTPUT_CANDIDATE_PATH, cleaned_rows)
REPORT_PATH.write_text(json.dumps(cleaning_report, indent=2), encoding='utf-8')

print(f'Wrote candidate cleaned CSV: {OUTPUT_CANDIDATE_PATH}')
print(f'Wrote cleaning report: {REPORT_PATH}')
display(pd.Series(cleaning_report, name='value').to_frame())


{'STRIP_COMMENTS': True, 'REMOVE_FILLING_ATTR': True, 'ROUND_FLOATS': True, 'FLOAT_DECIMALS': 3, 'COLLAPSE_WHITESPACE': True, 'TRAIN_MAX_LENGTH': 2048, 'NORMALIZE_MIDTERM_ROOT': True, 'ENFORCE_MIDTERM_GATE': True, 'STRICT_VIEWBOX': '0 0 256 256', 'RENDER_SIZE': 256, 'MAX_PATH_COUNT': 256, 'MAX_SVG_CHARS': 16000, 'CONFLICT_POLICY': 'keep_most_frequent', 'DEDUPLICATE_EXACT_PAIRS': True, 'FILTER_OVER_TOKEN_BUDGET': True, 'TOKEN_FILTER_MODEL_ID': 'Qwen/Qwen2.5-Coder-1.5B-Instruct', 'TOKEN_FILTER_MAX_LENGTH': 2048, 'TOKEN_SAMPLE_N': 2000, 'OUTPUT_COLUMNS': ['prompt', 'svg'], 'LOSS_MASKING_READY': True, 'TRAINING_OBJECTIVE': 'prompt conditioning with SVG-only loss masking on the svg target'}
Wrote candidate cleaned CSV: /content/drive/MyDrive/svg-kaggle-comptetition/datasets/train_canonicalized_nosvgo_len2048.csv
Wrote cleaning report: /content/drive/MyDrive/svg-kaggle-comptetition/datasets/train_canonicalized_nosvgo_len2048_report.json


,value
input_rows,50000
rows_changed_by_svg_cleaning,49220
rows_changed_by_midterm_root_normalization,50000
dropped_midterm_gate_rows,11
midterm_gate__parse_failed,10
dropped_long_svg_rows,8
midterm_gate__too_many_paths,1
dropped_rows_from_conflict_resolution,4067
dropped_exact_duplicate_pairs_after_cleaning,1
kept_rows,45913


## Candidate Diagnostics And Token Budget

This cell reruns diagnostics on the cleaned candidate CSV, verifies the midterm root-attribute contract, and samples tokenizer lengths against the configured training `max_length` so you can decide whether to keep a strict 2048-token filter off or turn it on.


In [10]:
candidate_results = run_diagnostics(OUTPUT_CANDIDATE_PATH)
candidate_token_stats = run_token_length_diagnostics(
    OUTPUT_CANDIDATE_PATH,
    sample_n=TOKEN_SAMPLE_N,
    max_length=TRAIN_MAX_LENGTH,
)


CSV: /content/drive/MyDrive/svg-kaggle-comptetition/datasets/train_canonicalized_nosvgo_len2048.csv


,value
csv_path,/content/drive/MyDrive/svg-kaggle-comptetition...
rows,39298
columns,"[prompt, svg]"
exact_duplicate_rows,0
conflicting_prompt_count,0
conflicting_rows,0
conflicting_row_pct,0.0
reused_svg_count,431
parse_fail,0
non_svg_root,0


,prompt,svg_chars,svg_prefix
0,A pixelated image of a face with a black and w...,3468,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
1,"A pixel art drawing of a flower, possibly repr...",3348,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
2,An icon with white soap bubbles floating on a ...,3153,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
3,A girl with red hair tied in pigtails with gre...,2898,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
4,An 8-bit style pixelated face with a startled ...,2815,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
5,A simple illustration of a multi-story buildin...,2793,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
6,A chess tower emoji is placed on top of a gree...,2719,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
7,The image depicts three pastel-colored cupcake...,2685,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
8,A server stack with a green checkmark indicati...,2679,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."
9,A simple clock icon with a blue face and hands...,2656,"<svg xmlns=""http://www.w3.org/2000/svg"" width=..."


,value
token_model_id,Qwen/Qwen2.5-Coder-1.5B-Instruct
token_sample_n,2000
token_p50,917
token_p95,1855
token_p99,2004
token_max,2043
over_max_length,0
over_max_length_pct,0.0
max_length,2048
